# The EP full-text library - Lesson 2
This notebook expands on lesson 1 to dive into more advanced concepts of EPAB, the implementation in TIP of the EP full-text library. We will introduce querying by full text fields, divisionals and parents, and search report fields. As we did in the first notebook, we first create an instance of the EPAB library. Remember that by default we are getting access to a test database

In [1]:
# Importing the EPAB client
from epo.tipdata.epab import EPABClient

# creating an instance of the EPAB client with the production database
epab = EPABClient(env='PROD')


## Querying by full text fields
Much like the [EP full-text search](https://www.epo.org/en/searching-for-patents/technical/ep-full-text), one of the most powerful features of the EPAB library is that it gives you access to the description, claims, title and abstract of the publications within the EPAB database. 

### Querying by the title
You can search for applications containing one or more terms in the title. When performing a first search for patent publications of a given technological concept, it is generally a good approach to search in the title, since when a publication contains the search term in the title it is likely that it is a good match for your search query. If you followed lesson 1, you probably can guess nomenclature of the search method: `query_title`.

In [2]:
# querying by the title of the publication with the word 'covid'
q = epab.query_title('covid')
q.get_results("title", limit=5, output_type='list')


[{'title': {'de': 'MIZELLENPRÄPARATE AUS VOLLSPEKTRUM-HANFÖL ZUR BEHANDLUNG VON TYP-II-DIABETES, ZUR VERRINGERUNG VON ENTZÜNDUNGEN WÄHREND COVID-19 UND ZUR VERBESSERUNG DER SCHLAFQUALITÄT',
   'en': 'MICELLE PREPARATIONS OF FULL-SPECTRUM HEMP OIL FOR TREATING TYPE II DIABETES, REDUCING INFLAMMATION DURING COVID-19, AND IMPROVING SLEEP QUALITY',
   'fr': "PRÉPARATIONS DE MICELLES D'HUILE DE CHANVRE À SPECTRE COMPLET POUR LE TRAITEMENT DU DIABÈTE DE TYPE II, LA RÉDUCTION DE L'INFLAMMATION PENDANT LE COVID-19 ET L'AMÉLIORATION DE LA QUALITÉ DU SOMMEIL"}},
 {'title': {'de': 'MULTIPLEX-COVID-19-VORHÄNGESCHLOSSTEST',
   'en': 'MULTIPLEXED COVID-19 PADLOCK ASSAY',
   'fr': 'TEST MULTIPLEXÉ PADLOCK POUR COVID-19'}},
 {'title': {'de': 'ANWENDUNG MESENCHYMALER STAMMZELLEN BEI DER HERSTELLUNG EINES ARZNEIMITTELS ZUR REPARATUR VON DURCH COVID-19 VERURSACHTEN LUNGENSCHÄDEN',
   'en': 'APPLICATION OF MESENCHYMAL STEM CELLS IN PREPARATION OF DRUG FOR REPAIRING LUNG DAMAGE CAUSED BY COVID-19',
   'fr'

#### Understanding fulltext languages
You can see in the result that the title field contains a dictionary with three titles. It is very important, when working with fulltext, to take into consideration that the EPO publishes the fulltext fields in the three official languages: German, English, and French.

When you search for a term in a fulltext field, by default you will search in all three languages. This can be problematic. A good example of a search query that would yield different results in English and German is the word "Gift."

In English, "gift" refers to a present or something given willingly to someone without payment. However, in German, "Gift" means "poison." You can change this by specifying one or more of the official languages with the strings `EN`, `DE` and `FR`.

In [3]:
# searching for publications with the word GIFT only in the English title
q = epab.query_title('gift', language="EN")
q.get_results("title", limit=5, )

,title.de,title.en,title.fr
0,"Kombination von Geschenk und Verpackung, insbe...",A combination comprising a gift and its casing...,"Combinaison d'un cadeau et de son emballage, e..."
1,GESCHENKVERPACKUNG / FLASCHENREGAL,GIFT PACKAGE/BOTTLE RACK,PAQUET-CADEAU / CASIER PORTE-BOUTEILLES
2,Heiratsgeschenkdose,A wedding gift box,Boîte pour cadeau de mariage
3,VERTEILUNGSVERWALTUNGSVERFAHREN FÜR GESCHENKGU...,GIFT CERTIFICATE DISTRIBUTION MANAGING METHOD ...,PROCEDE DE GESTION DE LA DISTRIBUTION DE BONS-...
4,VERFAHREN UND SYSTEM FÜR UNIVERSELLE GESCHENKR...,METHOD AND SYSTEM FOR UNIVERSAL GIFT REGISTRY,PROCEDE ET SYSTEME DE REGISTRE DE CADEAUX UNIV...


#### Refresher of query combination
We saw in lesson 1 that we can combine queries to create more complex queries. Let's see if there are any publications that contain the word gift in both the German and English titles. 

In [4]:
# we get a second query with publications mentioning poison, in German
r = epab.query_title('gift', language="DE")
print (f'publications with the word Gift in German', r)

#combining the two queries
s = q & r

print (f'Poisionus gifts found:', s)

publications with the word Gift in German 1581 publications
Poisionus gifts found: 0 publications


1581 publications
Poisionus gifts found: 

0 publications


### Case sensitivity
You have seen that we are querying in lowercase and the titles are displayed in all uppercase. It will come at no surprise that the search for full text terms is by default case insensitive. This can be overriden with `ignore_case=False`. Below we perform two queries with and without this parameter, to see the different results we get. 

In [5]:
# searching for publications with the word GIFT only in the English title ignoring case
q = epab.query_title('gift', language="EN")
print (f'Publications with the word gift in any combination of lower and upper case', q)

q.get_results('title', limit=5)




Publications with the word gift in any combination of lower and upper case 178 publications


,title.de,title.en,title.fr
0,"Kombination von Geschenk und Verpackung, insbe...",A combination comprising a gift and its casing...,"Combinaison d'un cadeau et de son emballage, e..."
1,GESCHENKVERPACKUNG / FLASCHENREGAL,GIFT PACKAGE/BOTTLE RACK,PAQUET-CADEAU / CASIER PORTE-BOUTEILLES
2,Heiratsgeschenkdose,A wedding gift box,Boîte pour cadeau de mariage
3,VERTEILUNGSVERWALTUNGSVERFAHREN FÜR GESCHENKGU...,GIFT CERTIFICATE DISTRIBUTION MANAGING METHOD ...,PROCEDE DE GESTION DE LA DISTRIBUTION DE BONS-...
4,VERFAHREN UND SYSTEM FÜR UNIVERSELLE GESCHENKR...,METHOD AND SYSTEM FOR UNIVERSAL GIFT REGISTRY,PROCEDE ET SYSTEME DE REGISTRE DE CADEAUX UNIV...


,title.de,title.en,title.fr
0,"Kombination von Geschenk und Verpackung, insbe...",A combination comprising a gift and its casing...,"Combinaison d'un cadeau et de son emballage, e..."
1,GESCHENKVERPACKUNG / FLASCHENREGAL,GIFT PACKAGE/BOTTLE RACK,PAQUET-CADEAU / CASIER PORTE-BOUTEILLES
2,Heiratsgeschenkdose,A wedding gift box,Boîte pour cadeau de mariage
3,VERTEILUNGSVERWALTUNGSVERFAHREN FÜR GESCHENKGU...,GIFT CERTIFICATE DISTRIBUTION MANAGING METHOD ...,PROCEDE DE GESTION DE LA DISTRIBUTION DE BONS-...
4,VERFAHREN UND SYSTEM FÜR UNIVERSELLE GESCHENKR...,METHOD AND SYSTEM FOR UNIVERSAL GIFT REGISTRY,PROCEDE ET SYSTEME DE REGISTRE DE CADEAUX UNIV...


In [6]:
# searching for publications with the word GIFT only in the English title forcing lowercase
r = epab.query_title('gift', language="EN", ignore_case=False)
print (f'Publications with the word gift in lowercase', r)

r.get_results('title', limit=5)

Publications with the word gift in lowercase 46 publications


,title.de,title.en,title.fr
0,"Zusammenspiel, Verfahren zum Zusammensetzen ei...","Puzzle, method of assembling a puzzle, gift an...","Puzzle, procédé d'assemblage d'un puzzle, cade..."
1,Behälter für Überraschungsgeschenke und ähnlic...,Container for surprise gifts and similar artic...,Récipient pour cadeaux surprise ou similaires ...
2,"Mehrschichtiges Material, insbesondere zur Her...","Material in sheet form, particularly for produ...","Matériau multicouche, particulièrement pour la..."
3,Universelle Geschenkschachtel,Versatile gift box,Boîte polyvalente à cadeaux
4,Dekorative Geschenkverpackung,Decorative gift package,Emballage décoratif pour cadeau


,title.de,title.en,title.fr
0,"Zusammenspiel, Verfahren zum Zusammensetzen ei...","Puzzle, method of assembling a puzzle, gift an...","Puzzle, procédé d'assemblage d'un puzzle, cade..."
1,Behälter für Überraschungsgeschenke und ähnlic...,Container for surprise gifts and similar artic...,Récipient pour cadeaux surprise ou similaires ...
2,"Mehrschichtiges Material, insbesondere zur Her...","Material in sheet form, particularly for produ...","Matériau multicouche, particulièrement pour la..."
3,Universelle Geschenkschachtel,Versatile gift box,Boîte polyvalente à cadeaux
4,Dekorative Geschenkverpackung,Decorative gift package,Emballage décoratif pour cadeau


### Multiple search terms
We can enter multiple search terms in the queries we run on EPAB by full text fields. When we enter multiple terms, by default these terms are combined with an `OR`

In [7]:
# Searching a set of possible terms (e.g. synonyms)
q = epab.query_title("covid, corona virus, coronavirus", language="EN")
print (q)
q.get_results("title.en", output_type="list", limit=10)

1225 publications


[{'title': {'en': 'Antigen binding polypeptides against spike glycoprotein (S2) of bovine coronavirus'}},
 {'title': {'en': 'NEUTRALIZING MONOCLONAL ANTIBODIES AGAINST SEVERE ACUTE RESPIRATORY SYNDROME-ASSOCIATED CORONAVIRUS'}},
 {'title': {'en': 'INHIBITORS OF CORONAVIRUS PROTEASE AND METHODS OF USE THEREOF'}},
 {'title': {'en': 'CANINE CORONAVIRUS VACCINE'}},
 {'title': {'en': 'CORONAVIRUS INACTIVATING AGENT'}},
 {'title': {'en': 'ANTIGENIC PEPTIDES OF SARS CORONAVIRUS AND USES THEREOF'}},
 {'title': {'en': 'REAGENTS AND METHODS FOR DETECTING SEVERE ACUTE RESPIRATORY SYNDROME CORONAVIRUS'}},
 {'title': {'en': 'INHIBITORS OF CORONAVIRUS'}},
 {'title': {'en': 'ANTIVIRAL AGENTS FOR THE TREATMENT, CONTROL AND PREVENTION OF INFECTIONS BY CORONAVIRUSES'}},
 {'title': {'en': 'PROPAGATION OF BOVINE CORONAVIRUS IN CHINESE HAMSTER OVARY CELLS'}}]

#### Multiple search terms combined with AND
We can also query with several strings, and specify that they all should be present, with the `match_all` parameter.

In [8]:
# We can also look for having multiple terms in the same title
q = epab.query_title("coronavirus, vaccine", match_all=True, language="EN")
print(q)
q.get_results("title.en", limit=5)

172 publications


,title.en
0,Canine coronavirus vaccine
1,CANINE CORONAVIRUS VACCINE
2,Canine coronavirus vaccine from feline enteric...
3,ANTI-CORONAVIRUS VACCINE
4,"Coronavirus, nucleic acid, protein and methods..."


#### Multiple search terms with advanced combinations
What if you want to mix `AND` with `OR` with the combinations of terms? Combining queries comes in handy for this case. 

In [9]:
# searching for synonims of Covid 
q = epab.query_title("covid, corona virus, coronavirus", language="EN")

# searching for synonims of vaccine
r = epab.query_title("vaccine%, inmun%", language="EN")

s = q & r

s.get_results('title.en', limit = 10)

,title.en
0,Vaccine against severe accute respiratory synd...
1,VACCINES AGAINST CORONAVIRUS AND METHODS OF USE
2,VACCINE COMPOSITION FOR PREVENTION OR TREATMEN...
3,VACCINE COMPOSITIONS FOR PREVENTING CORONAVIRU...
4,VACCINE WITH IMPROVED IMMUNOGENICITY AGAINST M...
5,VACCINE COMPOSITION FOR PREVENTING SEVERE ACUT...
6,VACCINE COMPOSITIONS FOR THE TREATMENT OF CORO...
7,VACCINE COMPOSITIONS FOR TREATING CORONAVIRUS ...
8,VACCINE AGAINST HUMAN-PATHOGENIC CORONAVIRUSES
9,VACCINE COMPOSITION AGAINST CORONAVIRUS


### Querying abstract, claims and description
You can query other parts of the fulltext such as the claims, the abstract, and the description with the same methods, obviously changing the part of the fulltext in the method nomenclature. 

In [10]:
# abstract search
q = epab.query_abstract("handover, base station", match_all=True, ignore_case=True)
print(q)
q.get_results("abstract", output_type="list", limit=2)

1512 publications


[{'abstract': {'language': 'EN',
   'text': '<p id="pa01" num="0001">A mobile communication system that is possible to shorten time from a handover request to a handover completion and to perform a high-speed handover is described. The mobile communication system includes a plurality of base stations which is configured to make wireless communication with a mobile terminal through, and a base station control apparatus connected to the base stations. The base station control apparatus includes a handover control section determines a handover base station to be handed over by the mobile terminal from the base stations, and directs the mobile terminal to hand over to the handover base station.\n<img id="iaf01" file="imgaf001.tif" wi="126" he="101" img-content="drawing" img-format="tif"/></p>'}},
 {'abstract': {'language': 'EN',
   'text': '<p id="pa01" num="0001">The present invention relates to maintaining an order of received data units during a handover procedure in a wireless communic

## Retrieving statistics from a query
Sometimes you will want to get statistics over the results of a query, before further processing it. The method `get_stats` returns a dataframe with the statistics over one or more selected fields. when you run this method on a query object, for the selected field(s) you will get the following information. 

- the `count` column reports the total number of occurrences of the corresponding field(s) value
- the `unique_publications` column reports the number of unique publications having that value
- the last two lines of the table are used to report the remainder and the total

### Statistics on patents about wireless communication networks
Let's look at an example. We will make a query for publications in the field of wireless communication networks, grouped in the CPC under H04W

In [11]:
# Running a query for all publications with CPC symbols starting with H04W
q = epab.query_ipc("H04W%")
q

219465 publications

In [12]:
# We want to see the distribution of the countries where the inventors mentioned in the publications resulting from the query live
q.get_stats("inventor.country")

,inventor.country,count,unique_publications
0,US,210044.0,62353.0
1,CN,146503.0,56188.0
2,KR,70361.0,20513.0
3,JP,53876.0,21717.0
4,SE,46623.0,17977.0
...,...,...,...
104,ZZ,1.0,1.0
105,EC,1.0,1.0
106,NP,1.0,1.0
107,GL,1.0,1.0


You can see that there are more inventors than publications. This happens because typically one application lists more than one inventor. We can also see what applicants are most active in the field of wireless communication networks

In [13]:
# We want to see the distribution of the countries where the inventors mentioned in the publications resulting from the query live
q.get_stats("applicant.name")

,applicant.name,count,unique_publications
0,"Huawei Technologies Co., Ltd.",20783.0,20735.0
1,Telefonaktiebolaget LM Ericsson (publ),12412.0,12400.0
2,"Samsung Electronics Co., Ltd.",11068.0,11044.0
3,Qualcomm Incorporated,8727.0,8726.0
4,LG Electronics Inc.,7262.0,7250.0
...,...,...,...
9836,"Teal Communications, Inc.",1.0,1.0
9837,"Sage, Gerald F.",1.0,1.0
9838,Numbereight Technologies Ltd,1.0,1.0
9839,"Mok, Ron",1.0,1.0


Again remember that a patent application can name more than one applicant, so it is possible that the sum of the `count` field will be higher than the sum of the `unique_publications` field.